In [0]:

SOURCE_TABLE = 'fraud_mlops.gold.production_drift_window'
OUTPUT_TABLE = 'fraud_mlops.gold.production_predictions'

FEATURES = [
    "amount",
    "log_amount",
    "type_cash_in",
    "type_cash_out",
    "type_debit",
    "type_payment",
    "type_transfer",
    "origin_txn_count_prior",
    "origin_avg_amount_prior",
    "origin_total_amount_prior",
    "amount_vs_origin_avg",
    "origin_has_history",
]


production_df = spark.read.table(SOURCE_TABLE).select('transaction_id',*FEATURES)


row_count = production_df.count()

unique_transactions = production_df.select('transaction_id').distinct().count()

print("Rows to score:", row_count)
print("Unique transaction IDs:", unique_transactions)

In [0]:
import json
import builtins
import requests

SCORING_URI = 'https://fraud-identify-endpoint.centralindia.inference.ml.azure.com/score'
ENDPOINT_KEY = dbutils.secrets.get(catalog="fraud_mlops",
                                   schema="gold",
                                   key="aml_endpoint_key")

production_pdf = production_df.toPandas()

BATCH_SIZE = 500

print("Transactions to score:", len(production_pdf))
print("Batch size:", BATCH_SIZE)
print("Number of batches:",(len(production_pdf) + BATCH_SIZE - 1) // BATCH_SIZE)

all_predictions = []
for start in range(0,len(production_pdf),BATCH_SIZE):
    batch = production_pdf.iloc[start:start+BATCH_SIZE]

    transaction_ids = batch['transaction_id'].tolist()
    model_input = json.loads(batch[FEATURES].to_json(orient='records'))

    payload = {
        'input_data':model_input
    }

    response = requests.post(
        SCORING_URI,
         headers={
            'Authorization': f'Bearer {ENDPOINT_KEY}',
            'Content-Type': 'application/json'
        },
        json=payload,
        timeout =120
    )

    response.raise_for_status()
    result = response.json()

    probabilities = result['fraud_probability']
    predictions = result['fraud_prediction']

    if not (len(transaction_ids)== len(probabilities)== len(predictions)):
        raise ValueError("Endpoint returned a different number of predictions than inputs.")

    for transaction_id,probability,prediction in zip(transaction_ids,probabilities,predictions):
        all_predictions.append((transaction_id,float(probability),int(prediction)))

    print(
        f"Scored {builtins.min(start + BATCH_SIZE, len(production_pdf))}"
        f" / {len(production_pdf)}"
    )





In [0]:
print("Predictions received:", len(all_predictions))

In [0]:
from pyspark.sql.functions import *
predictions_df = spark.createDataFrame(all_predictions,['transaction_id','fraud_probability','fraud_prediction'])

predictions_df = predictions_df.withColumn('model_version',lit('v2'))\
    .withColumn('deployment_name',lit('blue'))\
        .withColumn('prediction_timestamp',current_timestamp())

predictions_df.write.format('delta').mode('overwrite').saveAsTable(OUTPUT_TABLE)
    




In [0]:
df = spark.read.table('fraud_mlops.gold.production_predictions')
display(df)